[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-03.ipynb)

# 3주차 실습: 효율적 미세조정(PEFT) - LoRA

**목표.** 사전학습 한국어 분류 모델에 **LoRA 어댑터**를 붙여 미세조정하고, **학습 파라미터 수**가 전체 미세조정 대비 얼마나 줄었는지 출력으로 확인한다. 랭크 값을 하나 바꿔 학습 파라미터 수와 성능이 어떻게 변하는지 본다.

이 실습은 **과제 1**의 출발점이다. 과제 1에서는 이 코드를 그대로 돌리고, 결과를 제공된 비교표에 위치시켜 **도내 소규모 조직이 감당할 메모리·시간 예산** 안에서 선택 근거를 대야 한다.


## 제출 정보와 실험 설계

- 학번: 202221015
- 가정: 도내 소규모 관광 조직의 한국어 후기 분류용 사전 실험. 유료 서비스 없이 이미 보유한 8GB GPU 한 장, 기법당 학습 30분, GPU 메모리 8GB 이내 예산.
- 데이터: 강의 골격이 지정한 공개 NSMC 영화 리뷰. 제주 관광 데이터의 실측 결과로 해석하지 않으며, 실제 관광 후기 도메인으로의 일반화는 별도 검증 대상.
- 비교: 원래 LoRA r=4, TODO를 바꾼 r=8, 전체 미세조정. 동일한 사전학습 가중치, 분류 헤드 초기화 seed=42, 동일 분할, 1 epoch, batch=16, max_length=128, 학습률 5e-5, float32 조건.
- TODO 변경은 r=4 -> 8 한 곳. 원본대로 alpha=8을 유지하므로 alpha/r은 2 -> 1로 함께 달라짐. 따라서 순수한 표현 용량 효과만 분리한 실험은 아님.
- 추가 셀은 재현성, 실제 측정, 전체 미세조정 기준선과 비교표 작성용. GPU가 없으면 같은 코드가 CPU에서 실행되지만, 시간 예산 충족 여부는 다시 측정해야 함.
- AI 지원: 코드 작성과 오류 수정, 결과 정리에 생성형 AI 보조 사용. 아래 수치는 이 노트북의 실제 실행 결과이며, 독립적인 학습자 이해도 검증을 의미하지 않음.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다.


In [1]:
# 필요한 것 설치. 이미 준비된 실행 환경은 변경하지 않음.
import importlib.util
required = {"transformers": "transformers==4.57.6", "datasets": "datasets==4.4.1",
            "peft": "peft==0.18.1", "accelerate": "accelerate==1.12.0",
            "psutil": "psutil>=5.9", "numpy": "numpy>=1.26,<3"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    get_ipython().run_line_magic("pip", "-q install " + " ".join('"' + item + '"' for item in missing))
else:
    print("필요한 라이브러리가 설치되어 있음. 실제 버전은 다음 셀에 기록.")

필요한 라이브러리가 설치되어 있음. 실제 버전은 다음 셀에 기록.


In [2]:
import gc
import json
import os
import platform
import sys
import threading
import time
from pathlib import Path

import numpy as np
import psutil
import torch
import transformers
import datasets
import peft
import accelerate
from transformers import set_seed

SEED = 42
MODEL_REVISION = "68b30cd259f34a4b5aa8786392612ba2a2617fcc"
torch.set_num_threads(min(4, os.cpu_count() or 1))
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
set_seed(SEED)
RESULTS = []
RUNTIME = {
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": transformers.__version__, "datasets": datasets.__version__,
    "peft": peft.__version__, "accelerate": accelerate.__version__,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "gpu_total_gib": torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else None,
    "seed": SEED, "precision": "float32", "cpu_threads": torch.get_num_threads(),
}
print(json.dumps(RUNTIME, ensure_ascii=False, indent=2))

C:\Users\User\Desktop\School2\tmp\deepnlp_20260930\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "python": "3.10.0",
  "torch": "2.8.0+cu128",
  "transformers": "4.57.6",
  "datasets": "4.4.1",
  "peft": "0.18.1",
  "accelerate": "1.12.0",
  "device": "NVIDIA GeForce RTX 5060",
  "gpu_total_gib": 7.95953369140625,
  "seed": 42,
  "precision": "float32",
  "cpu_threads": 4
}


## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.


### 1-1. 데이터 준비

한국어 영화 리뷰 **감성 분류**(긍정/부정) 데이터셋 NSMC를 사용한다. 원본 저장소(e9t/nsmc)의 파일을 직접 읽고, Colab 무료 티어에서 빠르게 돌도록 소량만 쓴다.


In [3]:
from datasets import load_dataset

# NSMC 원본(e9t/nsmc GitHub)을 CSV로 직접 읽는다. 허브의 nsmc 는 스크립트 데이터셋이라 최신 datasets 에서 지원이 끊겼다
data_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
dataset = load_dataset("csv", data_files=data_url, sep="\t", split="train[:3000]")

# 원본의 결측/빈 문서는 토크나이징 전에 제외. 임의 문장으로 대체하지 않음.
raw_count = len(dataset)
dataset = dataset.filter(lambda row: isinstance(row["document"], str) and bool(row["document"].strip()))
removed_count = raw_count - len(dataset)
print("원본 선택:", raw_count, "결측/빈 문서 제외:", removed_count, "유효:", len(dataset))

# train/eval 로 나눈다
split = dataset.train_test_split(test_size=0.2, seed=42)
train_ds, eval_ds = split["train"], split["test"]

print("train:", len(train_ds), "eval:", len(eval_ds))
print("예시:", train_ds[0]["document"], "->", train_ds[0]["label"])

from collections import Counter
import hashlib
data_sha256 = hashlib.sha256(json.dumps(dataset.to_list(), ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
print("선택 데이터 SHA256:", data_sha256)
print("라벨 분포(train/eval):", dict(Counter(train_ds["label"])), dict(Counter(eval_ds["label"])))
shared_documents = len(set(train_ds["document"]) & set(eval_ds["document"]))
print("train/eval 공통 문서 문자열 수:", shared_documents)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 60000 examples [00:00, 436434.62 examples/s]

Generating train split: 120000 examples [00:00, 445890.46 examples/s]

Generating train split: 150000 examples [00:00, 442257.99 examples/s]

Filter:   0%|          | 0/3000 [00:00<?, ? examples/s]

Filter: 100%|██████████| 3000/3000 [00:00<00:00, 299928.78 examples/s]

원본 선택: 3000 결측/빈 문서 제외: 0 유효: 3000
train: 2400 eval: 600
예시: 사실여부를 떠나,알고왔던 아더와 너무 매칭이 안돼더라.원탁기사중 실제 검술 최고수는 랜슬롯으로 알고 있는데,트리스탄보다 못하고,싸우는 검술은 마치 중국검술 흉내낸거 같은게;; 그리고 란슬롯이 실제는 쌍검였나?너무 매칭이 안대 하튼 ㅋ기네비어역도 미스. -> 0
선택 데이터 SHA256: a110c485709daa66e913dc1c090bf7f5bcc21adf58e92e2207ea72a2a70e0239
라벨 분포(train/eval): {0: 1173, 1: 1227} {0: 306, 1: 294}
train/eval 공통 문서 문자열 수: 4


### 1-2. 토크나이징


In [4]:
from transformers import AutoTokenizer

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name, revision=MODEL_REVISION)

def tokenize(batch):
    return tokenizer(batch["document"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["document"])
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=["document"])
train_ds = train_ds.rename_column("label", "labels")
eval_ds = eval_ds.rename_column("label", "labels")
train_ds.set_format("torch")
eval_ds.set_format("torch")

print(train_ds.column_names)

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:  83%|████████▎ | 2000/2400 [00:00<00:00, 12971.91 examples/s]

Map: 100%|██████████| 2400/2400 [00:00<00:00, 12421.83 examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map: 100%|██████████| 600/600 [00:00<00:00, 12000.53 examples/s]

['id', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


### 1-3. 사전학습 모델 로딩

2주차에서 로딩한 것과 같은 한국어 분류 모델이다. 먼저 **전체 파라미터 수**를 확인해 둔다.


In [5]:
from transformers import AutoModelForSequenceClassification

set_seed(SEED)  # 모든 비교의 분류 헤드 초기화 조건 고정
model = AutoModelForSequenceClassification.from_pretrained(model_name, revision=MODEL_REVISION, num_labels=2)
total_params = sum(p.numel() for p in model.parameters())
print(f"모델 전체 파라미터 수: {total_params:,}")

print("모델 리비전:", model.config._commit_hash)

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


모델 전체 파라미터 수: 112,922,882
모델 리비전: 68b30cd259f34a4b5aa8786392612ba2a2617fcc


### 1-4. 전체 미세조정 기준 학습 파라미터 수

**전체 미세조정(full fine-tuning)** 은 모델의 모든 파라미터를 학습시킨다. 즉 학습 파라미터 수 = 전체 파라미터 수다.


In [6]:
full_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 미세조정 시 학습 파라미터 수: {full_trainable:,}")


전체 미세조정 시 학습 파라미터 수: 112,922,882


### 1-5. LoRA 어댑터 붙이기

모델을 **동결(freeze)** 하고, 저랭크 분해 행렬(어댑터)만 학습시킨다. `peft` 라이브러리가 어댑터를 붙이고, 학습 파라미터 수를 세어 준다.


In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,                     # 저랭크 차원
    lora_alpha=8,            # 스케일링 계수 (보통 r 의 2배)
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "value"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


trainable params: 739,586 || all params: 113,662,468 || trainable%: 0.6507


### 1-6. LoRA 미세조정 실행

Trainer로 어댑터만 학습시킨다. 몇 분 안에 끝난다.


In [8]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

def run_experiment(experiment_model, name):
    set_seed(SEED)
    training_args = TrainingArguments(
        output_dir=f"./results/{name}", num_train_epochs=1,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        learning_rate=5e-5, weight_decay=0.0, optim="adamw_torch",
        eval_strategy="no", save_strategy="no", logging_steps=50,
        report_to=[], seed=SEED, data_seed=SEED,
        fp16=False, bf16=False, tf32=False,
        dataloader_num_workers=0, dataloader_pin_memory=torch.cuda.is_available(),
        disable_tqdm=True,
    )
    trainable = sum(p.numel() for p in experiment_model.parameters() if p.requires_grad)
    wrapped_total = sum(p.numel() for p in experiment_model.parameters())
    trainer = Trainer(model=experiment_model, args=training_args,
                      train_dataset=train_ds, eval_dataset=eval_ds,
                      compute_metrics=compute_metrics)
    process = psutil.Process()
    cpu_rss = [process.memory_info().rss]
    stop = threading.Event()
    def sample_rss():
        while not stop.wait(0.02):
            cpu_rss.append(process.memory_info().rss)
    sampler = threading.Thread(target=sample_rss, daemon=True)
    sampler.start()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    try:
        train_result = trainer.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        train_seconds = time.perf_counter() - started
        train_gpu_peak = torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else None
        train_gpu_reserved = torch.cuda.max_memory_reserved() / 2**20 if torch.cuda.is_available() else None
        metrics = trainer.evaluate()
        cpu_rss.append(process.memory_info().rss)
        row = {
            "method": name, "trainable_parameters": trainable,
            "base_parameters": total_params, "wrapped_parameters": wrapped_total,
            "base_trainable_percent": 100 * trainable / total_params,
            "wrapped_trainable_percent": 100 * trainable / wrapped_total,
            "train_seconds": train_seconds, "trainer_train_seconds": train_result.metrics["train_runtime"],
            "train_loss": train_result.training_loss, "eval_accuracy": metrics["eval_accuracy"],
            "eval_loss": metrics["eval_loss"], "gpu_train_peak_allocated_mib": train_gpu_peak,
            "gpu_train_peak_reserved_mib": train_gpu_reserved,
            "cpu_process_peak_rss_mib_train_and_eval": max(cpu_rss) / 2**20,
            "train_examples": len(train_ds), "eval_examples": len(eval_ds),
            "steps": trainer.state.global_step, "seed": SEED,
        }
        RESULTS.append(row)
        print(json.dumps(row, ensure_ascii=False, indent=2))
        return row
    finally:
        stop.set()
        sampler.join(timeout=1)
        del trainer

def release_model():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("측정 기준: 학습 벽시계 시간은 train() 진입부터 GPU 동기화까지. 평가/다운로드 시간 제외.")
print("GPU 메모리는 학습 구간 PyTorch peak allocated/reserved. CPU RSS는 학습+평가 프로세스 전체의 20ms 표본 최대값.")

측정 기준: 학습 벽시계 시간은 train() 진입부터 GPU 동기화까지. 평가/다운로드 시간 제외.
GPU 메모리는 학습 구간 PyTorch peak allocated/reserved. CPU RSS는 학습+평가 프로세스 전체의 20ms 표본 최대값.


In [9]:
# 원본 Trainer 학습을 같은 하이퍼파라미터의 측정 함수로 감싼다.
baseline_lora = run_experiment(peft_model, "lora-r4")
del peft_model, model
release_model()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


{'loss': 0.6926, 'grad_norm': 0.8998391628265381, 'learning_rate': 3.366666666666667e-05, 'epoch': 0.3333333333333333}


{'loss': 0.6892, 'grad_norm': 0.9309131503105164, 'learning_rate': 1.7000000000000003e-05, 'epoch': 0.6666666666666666}


{'loss': 0.6858, 'grad_norm': 0.6202707886695862, 'learning_rate': 3.3333333333333335e-07, 'epoch': 1.0}
{'train_runtime': 15.6573, 'train_samples_per_second': 153.283, 'train_steps_per_second': 9.58, 'train_loss': 0.6891838836669922, 'epoch': 1.0}


{'eval_loss': 0.6866675019264221, 'eval_accuracy': 0.5783333333333334, 'eval_runtime': 1.8821, 'eval_samples_per_second': 318.786, 'eval_steps_per_second': 20.19, 'epoch': 1.0}
{
  "method": "lora-r4",
  "trainable_parameters": 739586,
  "base_parameters": 112922882,
  "wrapped_parameters": 113662468,
  "base_trainable_percent": 0.6549478607887461,
  "wrapped_trainable_percent": 0.6506862054059921,
  "train_seconds": 15.82471469999291,
  "trainer_train_seconds": 15.6573,
  "train_loss": 0.6891838836669922,
  "eval_accuracy": 0.5783333333333334,
  "eval_loss": 0.6866675019264221,
  "gpu_train_peak_allocated_mib": 1709.421875,
  "gpu_train_peak_reserved_mib": 1742.0,
  "cpu_process_peak_rss_mib_train_and_eval": 1754.66015625,
  "train_examples": 2400,
  "eval_examples": 600,
  "steps": 150,
  "seed": 42
}


## 2. 한 지점만 바꿔 보기

아래 셀의 `# TODO` 로 표시된 **한 곳(랭크 r)** 만 바꾸고 다시 실행하세요.

> 바꾸기 전 학습 파라미터 수와 평가 정확도를 먼저 적어 두면 무엇이 달라졌는지 비교할 수 있습니다.


In [10]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(model_name, revision=MODEL_REVISION, num_labels=2)

# TODO: 랭크 r 값을 바꿔 보세요 (예: 4 -> 8)
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "value"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,042 || all params: 113,809,924 || trainable%: 0.7794


In [11]:
changed_lora = run_experiment(peft_model, "lora-r8")
del peft_model, model
release_model()

{'loss': 0.6926, 'grad_norm': 0.9006465077400208, 'learning_rate': 3.366666666666667e-05, 'epoch': 0.3333333333333333}


{'loss': 0.6891, 'grad_norm': 0.9305049777030945, 'learning_rate': 1.7000000000000003e-05, 'epoch': 0.6666666666666666}


{'loss': 0.6857, 'grad_norm': 0.6219877004623413, 'learning_rate': 3.3333333333333335e-07, 'epoch': 1.0}
{'train_runtime': 16.1081, 'train_samples_per_second': 148.993, 'train_steps_per_second': 9.312, 'train_loss': 0.6891261037190756, 'epoch': 1.0}


{'eval_loss': 0.6866150498390198, 'eval_accuracy': 0.575, 'eval_runtime': 1.9373, 'eval_samples_per_second': 309.713, 'eval_steps_per_second': 19.615, 'epoch': 1.0}
{
  "method": "lora-r8",
  "trainable_parameters": 887042,
  "base_parameters": 112922882,
  "wrapped_parameters": 113809924,
  "base_trainable_percent": 0.7855290126229687,
  "wrapped_trainable_percent": 0.7794065480616611,
  "train_seconds": 16.345901100197807,
  "trainer_train_seconds": 16.1081,
  "train_loss": 0.6891261037190756,
  "eval_accuracy": 0.575,
  "eval_loss": 0.6866150498390198,
  "gpu_train_peak_allocated_mib": 1713.859375,
  "gpu_train_peak_reserved_mib": 1752.0,
  "cpu_process_peak_rss_mib_train_and_eval": 1867.01953125,
  "train_examples": 2400,
  "eval_examples": 600,
  "steps": 150,
  "seed": 42
}


### 2-1. 실제 전체 미세조정 기준선

비교표의 전체 미세조정 행도 같은 데이터와 초기화로 실제 학습. LoRA가 붙은 모델을 되돌리지 않고 원본 체크포인트를 새로 읽어 오염을 방지.

In [12]:
set_seed(SEED)
full_model = AutoModelForSequenceClassification.from_pretrained(model_name, revision=MODEL_REVISION, num_labels=2)
print("전체 미세조정 학습 파라미터 수:", sum(p.numel() for p in full_model.parameters() if p.requires_grad))
full_result = run_experiment(full_model, "full-finetuning")
del full_model
release_model()

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


전체 미세조정 학습 파라미터 수: 112922882


{'loss': 0.6227, 'grad_norm': 2.3868305683135986, 'learning_rate': 3.366666666666667e-05, 'epoch': 0.3333333333333333}


{'loss': 0.4265, 'grad_norm': 3.894455909729004, 'learning_rate': 1.7000000000000003e-05, 'epoch': 0.6666666666666666}


{'loss': 0.394, 'grad_norm': 7.20399284362793, 'learning_rate': 3.3333333333333335e-07, 'epoch': 1.0}
{'train_runtime': 23.0212, 'train_samples_per_second': 104.252, 'train_steps_per_second': 6.516, 'train_loss': 0.4810376230875651, 'epoch': 1.0}


{'eval_loss': 0.37357789278030396, 'eval_accuracy': 0.8416666666666667, 'eval_runtime': 1.9451, 'eval_samples_per_second': 308.471, 'eval_steps_per_second': 19.537, 'epoch': 1.0}
{
  "method": "full-finetuning",
  "trainable_parameters": 112922882,
  "base_parameters": 112922882,
  "wrapped_parameters": 112922882,
  "base_trainable_percent": 100.0,
  "wrapped_trainable_percent": 100.0,
  "train_seconds": 23.315944300033152,
  "trainer_train_seconds": 23.0212,
  "train_loss": 0.4810376230875651,
  "eval_accuracy": 0.8416666666666667,
  "eval_loss": 0.37357789278030396,
  "gpu_train_peak_allocated_mib": 2902.7529296875,
  "gpu_train_peak_reserved_mib": 3208.0,
  "cpu_process_peak_rss_mib_train_and_eval": 1874.53125,
  "train_examples": 2400,
  "eval_examples": 600,
  "steps": 150,
  "seed": 42
}


In [13]:
from IPython.display import Markdown, display

lookup = {row["method"]: row for row in RESULTS}
lines = [
    "| 기법 | 학습 파라미터 수 | 원본 전체 대비 비율 | GPU/런타임 | 학습 시간 | 평가 정확도 | 학습 GPU peak allocated / reserved |",
    "|---|---:|---:|---|---:|---:|---:|",
]
for key in ["full-finetuning", "lora-r8", "lora-r4"]:
    row = lookup[key]
    memory = (f'{row["gpu_train_peak_allocated_mib"]:.1f} / {row["gpu_train_peak_reserved_mib"]:.1f} MiB'
              if row["gpu_train_peak_allocated_mib"] is not None else "CPU 실행")
    lines.append(f'| {key} | {row["trainable_parameters"]:,} | {row["base_trainable_percent"]:.4f}% | {RUNTIME["device"]}, float32 | {row["train_seconds"]:.2f} s | {row["eval_accuracy"]:.4f} | {memory} |')
display(Markdown("\n".join(lines)))
print("평가셋 정답 수:", {key: round(row["eval_accuracy"] * len(eval_ds)) for key, row in lookup.items()})
print("CPU RSS 표본 최대(MiB, 학습+평가):", {key: round(row["cpu_process_peak_rss_mib_train_and_eval"], 1) for key, row in lookup.items()})
print("LoRA print_trainable_parameters의 분모는 어댑터/헤드 복사본을 포함한 모델. 위 표는 원본 total_params를 공통 분모로 사용.")

report = {"runtime": RUNTIME, "raw_count": raw_count, "removed_count": removed_count,
          "data_sha256": data_sha256, "shared_documents": shared_documents,
          "train_examples": len(train_ds), "eval_examples": len(eval_ds), "results": RESULTS}
if os.environ.get("WEEK03_METRICS_PATH"):
    Path(os.environ["WEEK03_METRICS_PATH"]).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

| 기법 | 학습 파라미터 수 | 원본 전체 대비 비율 | GPU/런타임 | 학습 시간 | 평가 정확도 | 학습 GPU peak allocated / reserved |
|---|---:|---:|---|---:|---:|---:|
| full-finetuning | 112,922,882 | 100.0000% | NVIDIA GeForce RTX 5060, float32 | 23.32 s | 0.8417 | 2902.8 / 3208.0 MiB |
| lora-r8 | 887,042 | 0.7855% | NVIDIA GeForce RTX 5060, float32 | 16.35 s | 0.5750 | 1713.9 / 1752.0 MiB |
| lora-r4 | 739,586 | 0.6549% | NVIDIA GeForce RTX 5060, float32 | 15.82 s | 0.5783 | 1709.4 / 1742.0 MiB |

평가셋 정답 수: {'lora-r4': 347, 'lora-r8': 345, 'full-finetuning': 505}
CPU RSS 표본 최대(MiB, 학습+평가): {'lora-r4': 1754.7, 'lora-r8': 1867.0, 'full-finetuning': 1874.5}
LoRA print_trainable_parameters의 분모는 어댑터/헤드 복사본을 포함한 모델. 위 표는 원본 total_params를 공통 분모로 사용.


## 3. 실무: 과제 1 안내

이 실습 코드는 **과제 1**의 밑바탕이다. 과제 1에서 할 일은 다음과 같다.

1. 한국어 데이터셋에서 **PEFT 기법 한 가지**(여기서는 LoRA)를 돌려본다
2. 강의 사이트에 제공된 **비교표**에서 자기 결과(학습 파라미터 수·메모리·시간·정확도)의 위치를 해석한다
3. **도내 소규모 조직이 감당할 메모리·시간 예산**을 조건으로 명시하고, 그 안에서 선택 근거를 댄다

채점은 3단계 부분점수(접근 설계 40 / 동작 40 / 성능·비교 20)로 이뤄진다. **1단계(왜 이 방법을 골랐는지)만 성립해도 부분 점수**를 받는다.

과제 1 상세와 비교표는 `assignments/week-03/` 폴더의 안내문을 참고한다.


## 4. 확인 질문

1. 1-5에서 출력된 LoRA 학습 파라미터 수는 전체 파라미터 수의 몇 퍼센트인가요? 손으로 계산해 보세요.
2. 랭크 r 을 키웠을 때 학습 파라미터 수는 어떻게 변하나요? 왜 그렇게 되는지 저랭크 근사의 의미와 연결지어 설명하세요.
3. 전체 미세조정 대비 LoRA는 메모리·시간·성능 측면에서 각각 무엇이 다른가요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.


## 확인 질문 답과 결과 해석

1. r=4의 학습 파라미터는 739,586개. 원본 전체 112,922,882개로 나누면 739,586 / 112,922,882 x 100 = 0.6549%. `print_trainable_parameters()`의 비율은 0.6507%이며, 이 출력은 LoRA 행렬과 학습용 분류 헤드 복사본을 포함한 113,662,468개를 분모로 사용하므로 차이가 있음.
2. r=4 -> 8에서 학습 파라미터는 739,586 -> 887,042개, 147,456개 증가. query/value 각각에 붙는 A, B의 크기가 r에 비례하므로 LoRA 행렬 자체는 147,456 -> 294,912개로 두 배. 함께 학습하는 분류 헤드 592,130개는 그대로여서 전체 학습 파라미터가 정확히 두 배는 아님. 저랭크 업데이트는 전체 768 x 768 행렬 대신 768 x r 및 r x 768 행렬의 곱으로 표현됨.
3. 전체 미세조정은 학습 23.32초, 정확도 84.17%, 학습 GPU 메모리 2902.8 / 3208.0 MiB. r=4는 15.82초, 57.83%, 1709.4 / 1742.0 MiB, r=8은 16.35초, 57.50%, 1713.9 / 1752.0 MiB. 동결한 원본 가중치의 경사/옵티마이저 상태를 줄여 메모리를 절약하지만, 원본 모델의 순전파와 일부 역전파 계산 및 활성값은 필요하므로 학습 파라미터 감소 배수만큼 메모리나 시간이 감소하지는 않음.

## 예산 안에서의 선택

- 과제에서 비교할 PEFT로 LoRA를 선택한 이유는 원본 가중치를 동결하고 작은 업데이트만 학습하여 메모리와 업데이트 저장량을 줄일 수 있기 때문. 후속 PEFT 실험의 기준은 `lora-r4`로 선택. 이번 실행의 정확도는 57.83%, 원본 대비 학습 파라미터는 0.6549%. r=8이 더 많은 학습 파라미터를 쓰면서도 개선을 보이지 않았으므로 낮은 랭크를 기준으로 삼되, 2문항 차이를 통계적으로 유의한 우위로 해석하지 않음.
- 전체 미세조정과 비교해 학습 GPU peak allocated는 41.1% 감소, 관측 학습 시간은 32.1% 감소. 원본 전체 가중치 대신 작은 어댑터와 분류 헤드만 학습하므로, 여러 적응 작업을 반복할 때의 메모리 여유와 저장할 업데이트 크기를 고려한 선택.
- 이번 실측에서 세 기법 모두 8GB/30분 예산 안에 들어왔음. 전체 미세조정은 84.17%로 LoRA보다 훨씬 높은 정확도를 보였으므로, 현재 조건에서 정확도를 우선하는 운영 후보는 전체 미세조정. "전체 미세조정이 불가능해서 LoRA가 필수" 또는 "LoRA가 동등한 정확도를 유지했다"는 결론은 성립하지 않음.
- LoRA는 비용 절감용 후속 실험 대상으로 유지하지만, 현재의 1 epoch/학습률 고정 결과만으로 운영 채택을 정당화할 수 없음. 학습률과 epoch를 조정하고 독립 test 및 여러 seed로 재평가하는 것이 다음 단계이며, 이 추가 실험을 수행했다고 주장하지 않음.
- TODO를 r=8로 바꾼 결과는 r=4 대비 정확도 -0.33%p. alpha=8을 고정한 원본 설정이라 alpha/r도 2 -> 1로 달라졌고, 한 seed만 사용했으므로 차이를 오직 랭크의 표현력 효과라고 단정할 수 없음.


## 재현 조건과 측정 한계

- NSMC 앞 3,000건에서 결측/빈 문서 0건 제외 후 train=2,400, eval=600. 데이터 분할 및 각 모델/분류 헤드 초기화 seed=42. 같은 eval을 모든 기법에 사용.
- train/eval에 같은 문서 문자열이 4개 있음. 골격의 분할을 유지했으므로 독립적인 최종 일반화 평가로 해석할 수 없음.
- 원본 모델 `monologg/koelectra-base-v3-discriminator`, 리비전 `68b30cd259f34a4b5aa8786392612ba2a2617fcc`. 1 epoch, batch=16, max_length=128, learning_rate=5e-5, AdamW, weight_decay=0, float32, query/value LoRA, alpha=8, dropout=0.1.
- 학습 시간은 `trainer.train()` 전후 동기화한 벽시계 실측. 모델/데이터 다운로드, 모델 로드 및 평가 시간 제외. 각 기법 1회 실행이므로 초기 CUDA 커널 준비, 실행 순서와 측정 변동의 영향이 포함됨.
- GPU 메모리는 학습 구간의 PyTorch peak allocated/reserved이며 GPU 전체 프로세스·드라이버 사용량과 다름. CPU RSS는 학습+평가 중 프로세스 전체를 20ms 간격으로 표본 측정한 최대값: full=1874.5 MiB, r4=1754.7 MiB, r8=1867.0 MiB. 독립 프로세스 측정이 아니므로 누적 캐시 영향이 있음.
- 같은 하이퍼파라미터를 고정한 소규모 비교이며, 각 기법의 최적 성능 비교나 여러 seed의 통계 검정은 수행하지 않음. 학습 데이터 일부로 만든 평가셋이고 별도의 NSMC 공식 test 평가도 아님.
- CPU fallback 가능하나 이 GPU의 속도/메모리 측정치는 CPU에서의 30분 예산 충족을 증명하지 않음.
- 환경: Python 3.10.0, torch 2.8.0+cu128, transformers 4.57.6, datasets 4.4.1, peft 0.18.1, accelerate 1.12.0.
- 선택 데이터 SHA256: `a110c485709daa66e913dc1c090bf7f5bcc21adf58e92e2207ea72a2a70e0239`.
- AI 지원: 코드 작성, 오류 수정, 결과 정리에 생성형 AI 보조 사용. 실행 수치는 저장된 노트북 출력에서 얻었으며 학습자 이해도는 별도로 확인할 사항.

출처: [공식 과제 1 안내](https://github.com/halla-ai/deepnlp-2026/blob/main/assignments/week-03/README.md), [공식 실습 골격](https://github.com/halla-ai/deepnlp-2026/blob/main/notebooks/week-03.ipynb), [NSMC 원본](https://github.com/e9t/nsmc).


## 5. 제출

이 노트북은 과제 1 이전의 **연습용**이다. 과제 1 답안은 별도 안내(비교표 포함)를 따라 제출한다.

연습 결과를 기록하려면 2주차와 같은 방법으로 `assignments/week-03/<내 학번>/` 에 올리면 된다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.
